#  Pure data case

We have three hyperparameters for the model:

- Relationship between $\sigma_s$ and $\sigma_0$
  - These follow the relation $\sigma_x^2 = \sigma_0^2\sigma_s^2$
  - We can set one of them to $c\sigma_{\epsilon}$, where $c$ is some constant deciding how you want to tune reconstruction penalty v/s regularisation
  - Or we can make them equal, and set both as: $\sigma_0 = \sigma_s = \sigma_x$
  - This removes $\sigma_{\epsilon}$ from the equation, do note that $\sigma_{\epsilon}$ is itself set with respect to $\sigma_x$, which we assume has 1 std.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import shelve
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import *
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
)
from pt_to_api import disjoint_ae, disjoint_ae_learned_sig
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
import numpy as np
from torch import nn
from torch import optim
import warnings
from dataclasses import dataclass
from typing import Any
import math
import gc
import pandas as pd
from pt_to_api import benchmark as B
import ast

MODE = "light"
SHELVE_CACHE_ROOT = Path.cwd() / "global-gauss-noise"
SHELVE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

## Helpers for plotting

In [ ]:
import matplotlib.lines as mlines


def plot_reconstruction_quality(df, key):
    """
    df must contain: dims, atom_ratio, n_sample_ratio, term3, and the column specified by key
    """
    dims_vals  = sorted(df["dims"].unique())
    atom_vals  = sorted(df["atom_ratio"].unique())
    term3_vals = sorted(df["term3"].unique())

    colors  = ["#2196F3", "#FF9800", "#4CAF50", "#E91E63", "#9C27B0"]
    markers = ["o", "s", "^", "D", "v"]

    color_map  = {a: colors[i] for i, a in enumerate(atom_vals)}
    marker_map = {a: markers[i] for i, a in enumerate(atom_vals)}

    n_rows = len(dims_vals)
    n_cols = len(term3_vals)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(5 * n_cols, 4 * n_rows),
        sharex=False,
        sharey="row",
        constrained_layout=True
    )
    if n_cols == 1 and n_rows == 1:
        axes = np.array([axes])
    if n_cols == 1:
        axes = axes.reshape(-1, 1)
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for row, dims in enumerate(dims_vals):
        for col, term3 in enumerate(term3_vals):
            sub = df[(df["dims"] == dims) & (df["term3"] == term3)]
            ax  = axes[row, col]

            for atom in atom_vals:
                d = sub[sub["atom_ratio"] == atom].sort_values("n_sample_ratio")
                ax.plot(
                    d["n_sample_ratio"], d[key],
                    marker=marker_map[atom], color=color_map[atom],
                    linewidth=2, markersize=7,
                    label=f"atom_ratio={atom}"
                )

            if row == 0:
                ax.set_title(f"term3 = {term3}", fontsize=12, fontweight="bold")
            if col == 0:
                ax.set_ylabel(f"dims={dims}\n{key}", fontsize=10)

            ax.set_xlabel("n_sample_ratio", fontsize=10)
            ax.grid(True, which="both", linestyle="--", alpha=0.4)
            ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
            ax.set_xticks(sorted(df["n_sample_ratio"].unique()))

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(atom_vals), fontsize=10,
               bbox_to_anchor=(0.5, -0.02), frameon=True)

    fig.suptitle(f"{key} vs. sample size\nby dims and term3", fontsize=14, fontweight="bold")

    plt.savefig(f"{key}_reconstruction_quality.png", dpi=150, bbox_inches="tight")
    plt.show()

def plot_mse_vs_meansim_multi(dfs, titles, suptitle, clip_quantile=None):
    n = len(dfs)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), constrained_layout=True)
    fig.suptitle(suptitle, fontweight="bold")

    if n == 1:
        axes = [axes]

    for ax, df, title in zip(axes, dfs, titles):
        ax.scatter(df["mse"], df["mean_sim"], s=60, alpha=0.7, color="#2196F3")
        ax.set_xlim(left=0, right=df["mse"].quantile(0.95))
        ax.set_xlabel("MSE ↓", fontsize=11)
        ax.set_ylabel("mean_sim ↑", fontsize=11)
        ax.set_title(title, fontsize=12)
        if clip_quantile:
            ax.set_xlim(left=0, right=df["mse"].quantile(clip_quantile))
        ax.grid(True, linestyle="--", alpha=0.4)

    plt.savefig("mse_vs_meansim_multi.png", dpi=150, bbox_inches="tight")
    plt.show()

# Start experiment

In [ ]:

def get_device(dim):
    if dim < 100:
        return "cpu"
    else:
        return "mps"



def get_metrics_df(metrics):
    mets = []
    for k, v in metrics.items():
        k = ast.literal_eval(k)
        m = v.copy()
        m["dims"] = k[0]
        m["atom_ratio"] = k[1]
        m["n_sample_ratio"] = k[2]
        m["noise_std"] = k[3]
        mets.append(m)
    
    df = pd.DataFrame(mets)
    df = df.drop(columns=["vec_sim"])
    
    return df

In [ ]:
def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]

def generate_synthetic_patches(
    patch_dim=72,
    n_components=10,
    k=3,
    n_samples=1000,
    noise_std=0.01,
    seed=42,
    sigma_x=1,
):
    rng = np.random.RandomState(seed)

    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    # each sample uses at most k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        k_i = rng.randint(1, k + 1)  # active atoms: 1..k
        idx = rng.choice(n_components, k_i, replace=False)
        codes_true[i, idx] = rng.randn(k_i)

    X = codes_true @ W_true

    scale = sigma_x / X.std()
    X *= scale
    W_true *= scale  # keeps codes_true @ W_true ≈ X
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition

In [ ]:
def run_single_test(dims_list, atoms_ratio, samples_ratio, noise_std_set, metrics):
    for dim in dims_list:
        for a in atoms_ratio:
            for sr in samples_ratio:
                for noise_std in noise_std_set:
                    try:
                        key = str((dim, a, sr, noise_std))
                        print("#######", key)
                        
                        atoms = math.ceil(dim*a)
                        k = atoms
                        n_samples = sr*atoms
                        if atoms == 1:
                            print("skip: atoms=1")
                            continue
                        if key in metrics:
                            print("skip: already done, delete the shelve file if you want to do it all over again")
                            continue
            
                        device = get_device(dim)
                        print("using", dim, atoms, k, n_samples, 0.01)
                        X, W_true, codes_true, dim_partition = generate_synthetic_patches(dim, atoms, k, n_samples=n_samples, noise_std=noise_std)
                
                        scaler = B.MeanPerDimGlobalStdScaler().fit(X)
                        X_scaled = scaler.transform(X)
        
                        mets = []
                        for run_idx in range(NUM_RUNS_PER_TEST):
                            print("RUN:", run_idx)
                            run = B.train(
                                X_scaled, atoms, 1e-2, epochs=4000, baseline_epochs=1000, device=device, init_strategy=B.SvdInitStrategy(), use_ln_term=False
                            )
                            mets.append(B.get_metrics_from_run(run, W_true))
                        gc.collect()
    
                        
                        metrics[key] = B.aggregate_metrics(mets)
                    except Exception as ex:
                        print("EXCEPTION, skipping", ex)

In [ ]:
# svd gives very deterministic results
NUM_RUNS_PER_TEST = 2

# original
DIMS_LIST = [10, 100]
ATOMS_RATIO = [0.1, 0.5, 0.9]    # for each dim, we test for 20%, 50% atoms for now, to understand how i can read the data
SAMPLES_RATIO = [20, 50, 100, 200]
NOISE_STD = [0.1, 0.2, 0.5, 0.7]


# test, comment for actual work
# DIMS_LIST = [10,]
# ATOMS_RATIO = [0.5, 0.9]    # for each dim, we test for 20%, 50% atoms for now, to understand how i can read the data
# SAMPLES_RATIO = [20]
# NOISE_STD = [0.1]


# Single sample

We will test a single sample here manually, 20% noise. The results suggest some interesting properties:

- SVD init gives better results
- `log` term is a problem with noise, it gives very noisy components (it tries to push away from 0 weights, but the signal is quite weak, so it ends up making components with a lot of noise). Although it still gives good results with SVD, there is more noise


As before, we see that increasing the data also gives good results.  


Note that we are now not able to reach the ideal MSE loss at all. I've tried multiple ways, have increased dataset size too, but we don't get close to what we get from baseline training or warmup.  

In [ ]:
dim = 100
a = 0.1
noise_std=0.7
sr=200

atoms = math.ceil(dim*a)
k = atoms
n_samples = sr*atoms

device = get_device(dim)
print("using", dim, atoms, k, n_samples, 0.01)
X, W_true, codes_true, dim_partition = generate_synthetic_patches(dim, atoms, k, n_samples=n_samples, noise_std=noise_std)

scaler = B.MeanPerDimGlobalStdScaler().fit(X)
X_scaled = scaler.transform(X)


In [ ]:
# ica

from sklearn.decomposition import FastICA

ica_estimator = FastICA(
    n_components=atoms, max_iter=100_000, whiten="arbitrary-variance", tol=15e-5
)
ica_estimator.fit(X_scaled)


In [ ]:
ica_estimator.mixing_.T
# metrics = B.get_metrics_from_run(run, W_true)
B.show_closest_component_of_W_for_each_component(ica_estimator.mixing_.T, W_true, (10,10), comps_per_row=1)

### SVD and no log term

In [ ]:
run = B.train(
    X_scaled, atoms, 1e-2, epochs=3000, baseline_epochs=2000, device=device, init_strategy=B.SvdInitStrategy(), use_ln_term=False
)

In [ ]:
# we see very promising resulst with svd, many of them have 99% similarity
metrics = B.get_metrics_from_run(run, W_true)
metrics["mean_sim"], metrics["vec_sim"]

In [ ]:
B.show_closest_component_of_W_for_each_component(run.components, W_true, (10,10))

### SVD and log term

In [ ]:
run = B.train(
    X_scaled, atoms, 1e-2, epochs=3000, baseline_epochs=2000, device=device, init_strategy=B.SvdInitStrategy(), use_ln_term=True
)

In [ ]:
metrics = B.get_metrics_from_run(run, W_true)
metrics["mean_sim"], metrics["vec_sim"]

In [ ]:
B.show_closest_component_of_W_for_each_component(run.components, W_true, (10,10))

### Normal init, no log

Quite a jumbled mess

In [ ]:
run = B.train(
    X_scaled, atoms, 1e-2, epochs=3000, baseline_epochs=2000, device=device, use_ln_term=False
)

In [ ]:
metrics = B.get_metrics_from_run(run, W_true)
metrics["mean_sim"], metrics["vec_sim"]

In [ ]:
B.show_closest_component_of_W_for_each_component(run.components, W_true, (10,10))

# Grid search

We will use SVD without log term now.  
Search across sample ratios to see how similarity changes with number of samples.  
With noisy data, we end up needing a lot more samples.  

In [ ]:
import math

# dims_list, atoms_ratio, samples_ratio, term3_set, kwarg_key, metrics

with shelve.open(SHELVE_CACHE_ROOT / "main2") as shelf:
    run_single_test(DIMS_LIST, ATOMS_RATIO, SAMPLES_RATIO, NOISE_STD, shelf)
    metrics = dict(shelf)


In [ ]:
df = get_metrics_df(metrics)
df.sort_values(by=["noise_std"])